# ENSO Snow BC: scratch notebook

The app's code lives in the `.py` modules; this notebook just imports them for exploring the data and trying out changes. Nothing here is used by the app, so it is fine to edit freely.

The first cell creates or updates the local database (`./data/snow.sqlite`, or wherever `SNOWAPP_DB` points). The first run downloads about 6 MB and takes under a minute; later runs only download what has changed.

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd

import datastore
import ingest
import snowdata
import snowdb
from snowmap import draw_station_map
from snowplot import snow_lineplot

con = snowdb.connect()
ingest.run_ingest(con)
snap = datastore.build_snapshot(con)
print(datastore.status_text(snap))

## Stations and current anomalies

In [ ]:
cols = ['name', 'elevation', 'active', 'n_seasons', 'span', 'obs_date', 'swe_mm', 'median_mm', 'pct_of_median', 'anom_status']
snap.stations[cols].sort_values('pct_of_median')

In [ ]:
draw_station_map(snap.stations[snap.stations['active']], show_anomaly=True)

## One station's water years

The same figure the app draws, for the RONI range the app opens with.

In [ ]:
sid = '3A25P'   # Squamish River Upper
lo, hi = snap.start_range
selected = snap.enso.index[(snap.enso >= lo) & (snap.enso <= hi)]
snow_lineplot(snap.matrices[sid], snap.complete[sid], selected, snap.current_wy,
              'rgba(255,110,95,0.3)', 'rgb(255,110,95)', f'{sid}, RONI {lo} to {hi}')

In [ ]:
# The season table behind the plot: one row per day of the water year, one column per water year
snap.matrices[sid].tail()

## Snow season indicators (Trends tab)

In [ ]:
from snowtrends import indicator_timeseries, indicator_vs_enso
import snowindicators as si

ind = snap.indicators[sid]   # days are days of the water year: 0 = 1 Oct; si.water_year_day_label() turns them into dates
display(ind.round(1).tail())
indicator_timeseries(ind, snap.enso, f'{sid} snow season indicators')

## ENSO (RONI) by water year

In [ ]:
snap.enso.tail(15)

## Querying the database directly

In [ ]:
pd.read_sql_query(
    "SELECT date, swe_mm, source FROM swe_daily WHERE station_id = '1A01P' ORDER BY date DESC LIMIT 10", con)

In [ ]:
snowdb.read_source_states(con)